# Customer Data Cleaning and Validation

## Goal

Review the raw Hillstrom customer file, apply the agreed cleaning decisions, document the findings, and save a cleaned CSV. The raw file is never modified.

## Scope and decisions

- Check missing values, exact repeated rows, data types, domains, category values, logical consistency, and outliers.
- Keep all rows. Flag exact repeated rows after their first occurrence because the dataset has no customer identifier.
- Keep valid extreme values. Do not trim or remove outliers.
- Correct the known ZIP category spelling in the cleaned copy.
- Do not encode categories, scale values, or resample here. Fit learned preprocessing steps only after the train/test split inside a pipeline.

## 1. Load the source file

The source and output paths below point to the project folders requested for this task.

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("../Data-raw/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")

print("Rows and columns:", data.shape)

Rows and columns: (64000, 12)


The file contains 64,000 rows and 12 source columns. Each row represents a customer record, but there is no unique customer ID. The campaign outcomes are retained for analysis; they are not all valid predictors for a pre-campaign score.

## 2. Inspect column types and missing values

In [3]:
print(data.dtypes)
print(data.isna().sum())

recency              int64
history_segment        str
history            float64
mens                 int64
womens               int64
zip_code               str
newbie               int64
channel                str
segment                str
visit                int64
conversion           int64
spend              float64
dtype: object
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64


No missing values or blank text values were found. No rows need to be removed or imputed for missingness.

## 3. Review categorical and target distributions

In [4]:
print(data["history_segment"].value_counts())
print("-"*50)
print(data["zip_code"].value_counts())
print("-"*50)
print(data["channel"].value_counts())
print("-"*50)
print(data["segment"].value_counts())

history_segment
1) $0 - $100        22970
2) $100 - $200      14254
3) $200 - $350      12289
4) $350 - $500       6409
5) $500 - $750       4911
6) $750 - $1,000     1859
7) $1,000 +          1308
Name: count, dtype: int64
--------------------------------------------------
zip_code
Surburban    28776
Urban        25661
Rural         9563
Name: count, dtype: int64
--------------------------------------------------
channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64
--------------------------------------------------
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


In [5]:
print(data["conversion"].value_counts().sort_index())
print("-"*50)
print((data["conversion"].value_counts(normalize=True).sort_index() * 100).round(2))

conversion
0    63422
1      578
Name: count, dtype: int64
--------------------------------------------------
conversion
0    99.1
1     0.9
Name: proportion, dtype: float64


In [6]:
print(data["recency"].describe())
print("-"*50)
print(data["history"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))
print("-"*50)
print(data["spend"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

count    64000.000000
mean         5.763734
std          3.507592
min          1.000000
25%          2.000000
50%          6.000000
75%          9.000000
max         12.000000
Name: recency, dtype: float64
--------------------------------------------------
count    64000.00
mean       242.09
std        256.16
min         29.99
25%         64.66
50%        158.11
75%        325.66
90%        561.19
95%        747.26
99%       1218.66
max       3345.93
Name: history, dtype: float64
--------------------------------------------------
count    64000.00
mean         1.05
std         15.04
min          0.00
25%          0.00
50%          0.00
75%          0.00
90%          0.00
95%          0.00
99%          0.00
max        499.00
Name: spend, dtype: float64


The conversion rate is about 0.9%, so the target classes are imbalanced. Accuracy alone will not be a reliable model-selection metric. The three campaign groups are close to equally sized.

## 4. Flag exact repeated rows

The flag marks only a later row that exactly matches a previous row across all source columns. It does not prove that the same person appears twice.

In [7]:
rows_in_repeated_groups = data.duplicated(keep=False).sum()
data["is_exact_repeat_after_first"] = data.duplicated(keep="first")

print("Additional exact repeated rows:", data["is_exact_repeat_after_first"].sum())
print("Rows in repeated groups:", rows_in_repeated_groups)
print(data["is_exact_repeat_after_first"].value_counts())

Additional exact repeated rows: 6562
Rows in repeated groups: 7634
is_exact_repeat_after_first
False    57438
True      6562
Name: count, dtype: int64


There are 6,562 exact repeated rows after the first occurrence; 7,634 rows belong to groups with repeated records. Keep every row because customer identity cannot be confirmed. Retain the flag for data-quality auditing and exclude it from model features.

## 5. Check value domains and logical consistency

In [8]:
binary_columns = ["mens", "womens", "newbie", "visit", "conversion"]
print(data[binary_columns].isin([0, 1]).all())
print("-"*50)
print("Recency outside 1-12 months:", (~data["recency"].between(1, 12)).sum())
print("Negative history values:", data["history"].lt(0).sum())
print("Negative spend values:", data["spend"].lt(0).sum())

mens          True
womens        True
newbie        True
visit         True
conversion    True
dtype: bool
--------------------------------------------------
Recency outside 1-12 months: 0
Negative history values: 0
Negative spend values: 0


In [9]:
print("Unexpected history_segment values:", (~data["history_segment"].isin([
    "1) $0 - $100", "2) $100 - $200", "3) $200 - $350",
    "4) $350 - $500", "5) $500 - $750",
    "6) $750 - $1,000", "7) $1,000 +"
])).sum())

print("Unexpected zip_code values:", (~data["zip_code"].isin([
    "Urban", "Surburban", "Rural"
])).sum())

print("Unexpected channel values:", (~data["channel"].isin([
    "Web", "Phone", "Multichannel"
])).sum())

print("Unexpected segment values:", (~data["segment"].isin([
    "Mens E-Mail", "Womens E-Mail", "No E-Mail"
])).sum())

Unexpected history_segment values: 0
Unexpected zip_code values: 0
Unexpected channel values: 0
Unexpected segment values: 0


### Check historical spend bands

Compare the numeric history value with the supplied history_segment label. The bands are interpreted as lower-inclusive and upper-exclusive.

In [10]:
history_labels = [
    "1) $0 - $100", "2) $100 - $200", "3) $200 - $350",
    "4) $350 - $500", "5) $500 - $750",
    "6) $750 - $1,000", "7) $1,000 +"
]
history_bins = [0, 100, 200, 350, 500, 750, 1000, np.inf]

expected_segment = pd.cut(
    data["history"], bins=history_bins,
    labels=history_labels, right=False
)
history_match = expected_segment.astype("string").eq(
    data["history_segment"].astype("string")
)

print("History band mismatches:", (~history_match).sum())

History band mismatches: 0


In [11]:
conversion_without_visit = (
    data["conversion"].eq(1) & data["visit"].eq(0)
).sum()
conversion_spend_mismatches = (
    (data["conversion"].eq(1) & data["spend"].le(0))
    | (data["conversion"].eq(0) & data["spend"].gt(0))
).sum()

print("Conversions without a visit:", conversion_without_visit)
print("Conversion/spend mismatches:", conversion_spend_mismatches)

Conversions without a visit: 0
Conversion/spend mismatches: 0


All binary columns contain only 0 or 1. Recency is within 1-12 months, amount fields are non-negative, categorical values are in the expected sets, history bands match their numeric values, and the observed conversion records have consistent visit and spend outcomes.

## 6. Review outliers without deleting them

The IQR rule is a review signal, not an automatic data-error test. Spend has many zeros, so this rule marks every positive spend value.

In [12]:
recency_q1 = data["recency"].quantile(0.25)
recency_q3 = data["recency"].quantile(0.75)
recency_iqr = recency_q3 - recency_q1
recency_outliers = (
    (data["recency"] < recency_q1 - 1.5 * recency_iqr)
    | (data["recency"] > recency_q3 + 1.5 * recency_iqr)
).sum()

history_q1 = data["history"].quantile(0.25)
history_q3 = data["history"].quantile(0.75)
history_iqr = history_q3 - history_q1
history_outliers = (
    (data["history"] < history_q1 - 1.5 * history_iqr)
    | (data["history"] > history_q3 + 1.5 * history_iqr)
).sum()

spend_q1 = data["spend"].quantile(0.25)
spend_q3 = data["spend"].quantile(0.75)
spend_iqr = spend_q3 - spend_q1
spend_outliers = (
    (data["spend"] < spend_q1 - 1.5 * spend_iqr)
    | (data["spend"] > spend_q3 + 1.5 * spend_iqr)
).sum()

print("IQR flags - recency:", recency_outliers)
print("IQR flags - history:", history_outliers)
print("IQR flags - spend:", spend_outliers)

IQR flags - recency: 0
IQR flags - history: 3593
IQR flags - spend: 578


The IQR rule flags 3,593 history values and 578 spend values. These are plausible historical purchases and positive outcomes, not known data errors. No outlier is removed or capped.

## 7. Correct the known category spelling

In [13]:
data["zip_code"] = data["zip_code"].replace({"Surburban": "Suburban"})
print(data["zip_code"].value_counts())

zip_code
Suburban    28776
Urban       25661
Rural        9563
Name: count, dtype: int64


Only the cleaned copy is changed. The raw source continues to use its original value.

## 8. Optimize in-memory data types

In [14]:
memory_before_mb = data.memory_usage(deep=True).sum() / (1024 ** 2)

data["recency"] = data["recency"].astype("int8")
data["history_segment"] = data["history_segment"].astype("category")
data["mens"] = data["mens"].astype("int8")
data["womens"] = data["womens"].astype("int8")
data["zip_code"] = data["zip_code"].astype("category")
data["newbie"] = data["newbie"].astype("int8")
data["channel"] = data["channel"].astype("category")
data["segment"] = data["segment"].astype("category")
data["visit"] = data["visit"].astype("int8")
data["conversion"] = data["conversion"].astype("int8")

cleaned_data = data.copy()
memory_after_mb = cleaned_data.memory_usage(deep=True).sum() / (1024 ** 2)

print("Memory before type optimization (MB):", round(memory_before_mb, 2))
print("Memory after type optimization (MB):", round(memory_after_mb, 2))

Memory before type optimization (MB): 8.09
Memory after type optimization (MB): 1.65


Categorical columns and binary flags use smaller in-memory types. The CSV format does not retain pandas category metadata, so these casts are applied again whenever the CSV is loaded.

## 9. Save and verify the cleaned file

Keep all 64,000 rows and all original columns, plus the audit flag. Do not use the audit flag as a model predictor.

In [15]:
cleaned_data = data
cleaned_data.to_csv("../Data-raw/Cleaned/hillstrom_customers_cleaned.csv", index=False)

saved_data = pd.read_csv("../Data-raw/Cleaned/hillstrom_customers_cleaned.csv")

print("Saved rows and columns:", saved_data.shape)
print("Saved repeat flags:", saved_data["is_exact_repeat_after_first"].sum())
print("-"*50)
print("Corrected Suburban values:", saved_data["zip_code"].eq("Suburban").sum())
print("Remaining Surburban values:", saved_data["zip_code"].eq("Surburban").sum())
print("Saved file:", "../Data-raw/Cleaned/hillstrom_customers_cleaned.csv")

Saved rows and columns: (64000, 13)
Saved repeat flags: 6562
--------------------------------------------------
Corrected Suburban values: 28776
Remaining Surburban values: 0
Saved file: ../Data-raw/Cleaned/hillstrom_customers_cleaned.csv


## Final decisions

- No missing values were imputed and no rows were removed.
- Exact repeated rows were kept and flagged for auditing.
- Valid outliers were kept.
- The known ZIP category spelling was corrected in the cleaned copy.
- Domain, categorical, historical-band, and outcome-consistency checks were completed.
- One-hot encoding and scaling remain for the model pipeline after the train/test split.
- The cleaned CSV is saved under Data-raw/Cleaned.

## Limitation

There is no customer identifier. The repeat flag identifies identical records after their first occurrence; it does not establish that they belong to the same customer.